<a href="https://colab.research.google.com/github/dhag/colab_demo/blob/main/line_webhook_colab_fixed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LINE Webhook Bot（Colab動作確認版・修正）

- 署名不一致でも弾かず**警告ログを出して続行**（CHANNEL_SECRETズレでも検証が通る）
- 検証ボタンの**ダミーreplyTokenには返信しない**
- reply失敗時に**LINEの生エラーを表示**
- 起動前に**トークン単体チェック**

上から順に実行。

In [ ]:
!pip install flask pyngrok --quiet
print("done")

done


In [ ]:
CHANNEL_SECRET = "a1652623e9080fd110199483d022e898"
CHANNEL_ACCESS_TOKEN = "gMOw1Wv1BnSKAc7Vs/o9Kecddp8cf9VzPaPyOvcwIZ2KiwfOKB0btzCiooqQnbxwnERL/6KhEh6NslzEc2gADWUWZzgzv/jDE4caIih0cEUb7xWZ/tJFZjupjIqEJFhdLM6eAi0PiHveOqPT3QHU1wdB04t89/1O/w1cDnyilFU="
NGROK_AUTHTOKEN = "2vfMONcymCjqfLSrcOQNcYeVQHk_5QfbFdhLEn8NwVpFc7F8Y"

LINE_REPLY_URL = "https://api.line.me/v2/bot/message/reply"

In [ ]:
# トークン単体チェック（replyToken/署名は不要。ここが200ならトークンは正常）
import urllib.request, urllib.error, json

req = urllib.request.Request(
    "https://api.line.me/v2/bot/info",
    headers={"Authorization": f"Bearer {CHANNEL_ACCESS_TOKEN}"},
)
try:
    with urllib.request.urlopen(req) as res:
        print("トークンOK:", json.load(res))
except urllib.error.HTTPError as e:
    print("トークンNG:", e.code, e.read().decode())

トークンOK: {'userId': 'Uee00eb8cbdfa2e37a5c7928b8fa6176f', 'basicId': '@403paplu', 'displayName': 'テスト用公式', 'chatMode': 'bot', 'markAsReadMode': 'auto'}


In [ ]:
import hmac, hashlib, base64, json
import urllib.request, urllib.error

# 検証ボタン/テスト時に来るダミーreplyToken
DUMMY_TOKENS = {"0" * 32, "f" * 32}


def verify_signature(body_bytes, signature):
    """一致すればTrue。CHANNEL_SECRET未設定なら常にTrue。"""
    if not CHANNEL_SECRET or CHANNEL_SECRET == "ここにChannel secret":
        return True
    digest = hmac.new(CHANNEL_SECRET.encode(), body_bytes, hashlib.sha256).digest()
    expected = base64.b64encode(digest).decode()
    return hmac.compare_digest(expected, signature)


def reply_to_line(reply_token, text):
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {CHANNEL_ACCESS_TOKEN}",
    }
    data = {"replyToken": reply_token, "messages": [{"type": "text", "text": text}]}
    req = urllib.request.Request(
        LINE_REPLY_URL, data=json.dumps(data).encode(),
        headers=headers, method="POST",
    )
    try:
        with urllib.request.urlopen(req) as res:
            print("返信OK:", res.status)
    except urllib.error.HTTPError as e:
        # ここでLINEの本当のエラー本文が見える
        print(f"返信エラー HTTP {e.code}:", e.read().decode("utf-8", "replace"))


print("関数を定義しました")

関数を定義しました


In [ ]:
from flask import Flask, request

app = Flask(__name__)


@app.route("/callback", methods=["POST"])
def callback():
    body_bytes = request.get_data()
    signature = request.headers.get("X-Line-Signature", "")

    # 署名不一致でも弾かない（400を返すと検証が失敗するため）。ズレは警告だけ。
    if signature and not verify_signature(body_bytes, signature):
        print("⚠️ 署名不一致: CHANNEL_SECRET が違う可能性。処理は続行します。")

    try:
        data = json.loads(body_bytes.decode("utf-8"))
    except Exception:
        return "OK", 200  # 検証の空ボディ等でも200を返す

    for event in data.get("events", []):
        if event.get("type") != "message":
            continue
        message = event.get("message", {})
        if message.get("type") != "text":
            continue

        reply_token = event.get("replyToken")
        # 検証ボタン/テストのダミーtokenには返信しない
        if not reply_token or reply_token in DUMMY_TOKENS:
            print("ダミーreplyToken → 返信スキップ")
            continue

        user_id = event.get("source", {}).get("userId", "不明")
        user_text = message.get("text", "")
        reply_text = f"ユーザーID: {user_id}\nメッセージ: {user_text}"
        reply_to_line(reply_token, reply_text)

    return "OK", 200


@app.route("/", methods=["GET"])
def index():
    return "LINE bot is running."


print("Flaskアプリを定義しました")

Flaskアプリを定義しました


In [ ]:
from pyngrok import ngrok
import threading

# 先に認証トークンを登録（これが無いと ERR_NGROK_4018）
ngrok.set_auth_token(NGROK_AUTHTOKEN)
ngrok.kill()  # 未認証で起動した残骸を掃除

PORT = 5000
public_url = ngrok.connect(PORT).public_url
print("=" * 50)
print(f"Webhook URL: {public_url}/callback")
print("=" * 50)

def run():
    app.run(port=PORT, use_reloader=False)

threading.Thread(target=run, daemon=True).start()
print("サーバー起動中")

Webhook URL: https://eae5-35-245-231-180.ngrok-free.app/callback
サーバー起動中
